In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.types import NumericType

In [ ]:
spark = (
    SparkSession.builder
    .appName("ReadAMLCSV")
    .master("spark://100.127.25.114:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

path = "hdfs://100.127.25.114:9000/data/hotel_bookings.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path)
)

Xem size & shape

In [ ]:
row_count = df.count()
col_count = len(df.columns)

print("Rows:", row_count)
print("Columns:", col_count)

Rows: 119390
Columns: 32


Xem schema

In [9]:
df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

In [13]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required_car_parking_spaces|total_of_special_reque

In [ ]:
total_rows = df.count()

distinct_rows = df.distinct().count()

duplicate_rows_count = total_rows - distinct_rows
print(f"Số dòng bị lặp: {duplicate_rows_count}")

Số dòng bị lặp: 31994


Kiểm tra số dòng lặp chính xác

In [26]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

windowSpec = Window.partitionBy(df.columns).orderBy(df.columns[0])

df_with_row_num = df.withColumn("row_num", row_number().over(windowSpec))


two_duplicate_rows = df_with_row_num.filter(
    col("row_num").isin([1, 2])
).join(

    duplicates.select(df.columns), 
    on=df.columns, 
    how="inner"
)

two_duplicate_rows.drop("row_num").show()

+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+
|     hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|  customer_type|  adr|required_car_parking_spaces|tota

In [27]:
df = df.dropDuplicates()

In [29]:
df.groupBy("Hotel").count().orderBy(desc("count")).show()

+------------+-----+
|       Hotel|count|
+------------+-----+
|  City Hotel|53428|
|Resort Hotel|33968|
+------------+-----+



In [30]:
df.groupBy("arrival_date_month").count().orderBy(desc("count")).show()   

+------------------+-----+
|arrival_date_month|count|
+------------------+-----+
|            August|11257|
|              July|10057|
|               May| 8355|
|             April| 7908|
|              June| 7765|
|             March| 7513|
|           October| 6934|
|         September| 6690|
|          February| 6098|
|          December| 5131|
|          November| 4995|
|           January| 4693|
+------------------+-----+



In [31]:
df.groupBy("children").count().orderBy(desc("count")).show()   

+--------+-----+
|children|count|
+--------+-----+
|       0|79028|
|       1| 4695|
|       2| 3593|
|       3|   75|
|      NA|    4|
|      10|    1|
+--------+-----+



In [32]:
df.groupBy("babies").count().orderBy(desc("count")).show()   

+------+-----+
|babies|count|
+------+-----+
|     0|86482|
|     1|  897|
|     2|   15|
|    10|    1|
|     9|    1|
+------+-----+



In [36]:
df.groupBy("market_segment").count().orderBy(desc("count")).show()   

+--------------+-----+
|market_segment|count|
+--------------+-----+
|     Online TA|51618|
| Offline TA/TO|13889|
|        Direct|11804|
|        Groups| 4942|
|     Corporate| 4212|
| Complementary|  702|
|      Aviation|  227|
|     Undefined|    2|
+--------------+-----+



In [37]:
df.groupBy("distribution_channel").count().orderBy(desc("count")).show()   

+--------------------+-----+
|distribution_channel|count|
+--------------------+-----+
|               TA/TO|69141|
|              Direct|12988|
|           Corporate| 5081|
|                 GDS|  181|
|           Undefined|    5|
+--------------------+-----+



In [38]:
df.groupBy("reserved_room_type").count().orderBy(desc("count")).show() 

+------------------+-----+
|reserved_room_type|count|
+------------------+-----+
|                 A|56552|
|                 D|17398|
|                 E| 6049|
|                 F| 2823|
|                 G| 2052|
|                 B|  999|
|                 C|  915|
|                 H|  596|
|                 L|    6|
|                 P|    6|
+------------------+-----+



In [39]:
df.groupBy("assigned_room_type").count().orderBy(desc("count")).show() 

+------------------+-----+
|assigned_room_type|count|
+------------------+-----+
|                 A|46313|
|                 D|22432|
|                 E| 7195|
|                 F| 3627|
|                 G| 2498|
|                 C| 2165|
|                 B| 1820|
|                 H|  706|
|                 I|  357|
|                 K|  276|
|                 P|    6|
|                 L|    1|
+------------------+-----+



In [41]:
df.groupBy("deposit_type").count().orderBy(desc("count")).show() 

+------------+-----+
|deposit_type|count|
+------------+-----+
|  No Deposit|86251|
|  Non Refund| 1038|
|  Refundable|  107|
+------------+-----+



In [46]:
df.groupBy("reservation_status").count().orderBy(desc("count")).show() 

+------------------+-----+
|reservation_status|count|
+------------------+-----+
|         Check-Out|63371|
|          Canceled|23011|
|           No-Show| 1014|
+------------------+-----+



In [47]:
df.groupBy("customer_type").count().orderBy(desc("count")).show() 

+---------------+-----+
|  customer_type|count|
+---------------+-----+
|      Transient|71986|
|Transient-Party|11727|
|       Contract| 3139|
|          Group|  544|
+---------------+-----+



In [50]:
numeric_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(numeric_cols)

['is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


In [52]:
df.select(numeric_cols).describe().show()

+-------+-------------------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+--------------------+-------------------+----------------------+------------------------------+-------------------+--------------------+------------------+---------------------------+-------------------------+
|summary|        is_canceled|        lead_time| arrival_date_year|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|            adults|              babies|  is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|    booking_changes|days_in_waiting_list|               adr|required_car_parking_spaces|total_of_special_requests|
+-------+-------------------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+--------------------+----------------

Missing value

In [53]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required_car_parking_spaces|total_of_special_reque